# Du Lịch Tri Tôn — Master Data Cleaning Notebook (Pandas & GIS Alignment)
Notebook xử lý chuẩn hóa dữ liệu du lịch Tri Tôn sử dụng thư viện **Pandas**, kiểm tra tọa độ GIS Bounding Box `[10.25 - 10.55 Lat, 104.85 - 105.15 Lng]`, NFC unicode, loại bỏ cụm từ cấm 'Huyện Tri Tôn', đồng bộ Xã/Thị trấn và xuất file master.

In [ ]:
import pandas as pd
import unicodedata
import json
import os

# 1. Read raw CSV dataset
csv_path = '../data/tri_ton_master_cleaned.csv'
df = pd.read_csv(csv_path, encoding='utf-8-sig')
print(f'Initial records loaded: {len(df)}')
df.head()

In [ ]:
# 2. NFC Normalization & Administrative Ward Corrections
def clean_text_field(text):
    if not isinstance(text, str) or pd.isna(text):
        return ''
    text = unicodedata.normalize('NFC', text)
    text = text.replace('Huyện Tri Tôn, ', '').replace(', Huyện Tri Tôn', '').replace('Huyện Tri Tôn', '')
    text = text.replace('Xã Chau Lăng', 'Xã Châu Lăng')
    text = text.replace('Xã Cô Tô', 'Xã Núi Tô')
    return text.strip()

for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].apply(clean_text_field)

print('Text normalization completed.')

In [ ]:
# 3. GIS Bounding Box Filtering & Coordinate Validation
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

valid_bbox = (df['latitude'] >= 10.25) & (df['latitude'] <= 10.55) & \
             (df['longitude'] >= 104.85) & (df['longitude'] <= 105.15)

df_valid = df[valid_bbox].copy()
print(f'Valid GIS records: {len(df_valid)}')
df_valid[['id', 'name', 'commune', 'latitude', 'longitude']].head()

In [ ]:
# 4. Deduplication & Export Master Files
df_valid['norm_name'] = df_valid['name'].str.lower()
df_valid['round_lat'] = df_valid['latitude'].round(3)
df_valid['round_lng'] = df_valid['longitude'].round(3)

df_final = df_valid.drop_duplicates(subset=['norm_name', 'round_lat', 'round_lng'], keep='first').copy()
df_final.drop(columns=['norm_name', 'round_lat', 'round_lng'], inplace=True)

output_csv = '../data/tri_ton_master_cleaned.csv'
df_final.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f'Successfully saved cleaned dataset ({len(df_final)} records) to {output_csv}')